# AI Resume–Job Description Semantic Matching System

**Author**: Anshika Saklani  
**Project**: AI Resume–Job Description Semantic Matching System  
**Architecture**: Fine-Tuned DistilBERT (`distilbert-base-uncased`) Sequence Classification

---

### Executive Workflow
```text
Candidate Resume (PDF/DOCX/TXT) + Job Description
                      ↓
           Text Preprocessing & Normalization
                      ↓
       [CLS] resume_text [SEP] job_description [SEP]
                      ↓
        DistilBERT Transformer Encoder (6 Layers)
                      ↓
         Pooler / [CLS] Dense Classification Head
                      ↓
     Softmax Probabilities: [No Fit, Potential Fit, Good Fit]
                      ↓
   Model Match Score + Skill Gap Analysis + Recommendations
```



In [1]:
!pip install -q transformers datasets evaluate accelerate scikit-learn pandas numpy matplotlib seaborn pypdf python-docx plotly



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os, sys, random, json, re, unicodedata
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Compute Device:", device)
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))


Compute Device: cpu


## 1. Deep Educational Theory

### A. Tokenization
Before text can enter a Transformer, it must be converted into numerical representations called token IDs.
DistilBERT uses **WordPiece** tokenization, which breaks down words into subword units (e.g. `embeddings` -> `['em', '##bed', '##ding', '##s']`). This prevents Out-Of-Vocabulary (OOV) errors and handles technical jargon cleanly.
- `[CLS]` (ID: 101): Placed at the very start of every sequence. Its representation aggregates sentence-level context for classification.
- `[SEP]` (ID: 102): Placed between the resume and job description, signaling the boundary between the two texts.
- `[PAD]` (ID: 0): Padded to uniform length (`max_length=512`).

### B. Transformer Contextual Embeddings & Multi-Head Self-Attention
Unlike static embeddings (Word2Vec or GloVe) where "Python" has a fixed vector regardless of context, Transformer self-attention computes dynamic weights between every word pair in the combined sequence:
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$
This allows the model to understand that "predictive models using scikit-learn" in a resume strongly relates to "experience building machine-learning models" in a job description.

### C. Fine-Tuning & Classification Head
We adapt pre-trained `distilbert-base-uncased` to sequence pair classification. The `[CLS]` hidden state ($768$-dim) passes through a linear pre-classifier layer (with dropout and ReLU activation), followed by a classification projection layer outputting 3 logits corresponding to:
- `Class 0`: **No Fit**
- `Class 1`: **Potential Fit**
- `Class 2`: **Good Fit**


In [3]:
# 2. Dataset Loading & Exploration
ds = load_dataset("michaelozon/candidate-matching-synthetic", split="resumes")
raw_df = pd.DataFrame(ds)
print("Raw Resumes Shape:", raw_df.shape)
print("Columns:", raw_df.columns.tolist())
raw_df.head(2)


Raw Resumes Shape: (10000, 9)
Columns: ['resume_id', 'role', 'seniority', 'years_experience', 'industry', 'education', 'skills', 'summary', 'experience_bullets']


,resume_id,role,seniority,years_experience,industry,education,skills,summary,experience_bullets
0,R_000000,Software Engineer,Senior,12,EdTech,BSc,"[OOP, Databases, Git, Docker, Python, Unit Tes...",Software Engineer with 12 years of experience ...,[Delivered results using structured workflows ...
1,R_000001,Marketing Manager,Junior,2,E-commerce,BSc,"[Content Marketing, Meta Ads, Conversion Optim...",Marketing Manager with 2 years of experience i...,[Delivered results using structured workflows ...


In [4]:
# 3. Preprocessing and Zero-Leakage Dataset Construction
def clean_text(text):
    if text is None or not isinstance(text, str): return ""
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

print("Preprocessing and Pair Construction ready!")


Preprocessing and Pair Construction ready!


In [5]:
# 4. TF-IDF & Logistic Regression Baselines
print("Evaluating Baseline Models...")


Evaluating Baseline Models...


In [6]:
# 5. DistilBERT Fine-Tuning
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Tokenizer loaded successfully!")


Tokenizer loaded successfully!


In [7]:
# 6. Evaluation, Curves & Error Analysis
print("Evaluating on Test Set...")


Evaluating on Test Set...


In [8]:
# 7. Live Interactive Prediction Demo
print("Running Sample Inference Demo...")


Running Sample Inference Demo...
